In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Merepresentasikan komputer kuantum untuk transpiler


<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Untuk mengonversi Circuit abstrak menjadi ISA circuit yang bisa dijalankan di QPU (unit pemrosesan kuantum) tertentu, transpiler membutuhkan informasi tertentu tentang QPU tersebut. Informasi ini ditemukan di dua tempat: objek `BackendV2` (atau `BackendV1` yang lebih lama) yang akan kamu gunakan untuk mengirim job, dan atribut `Target` dari backend.

- [`Target`](../api/qiskit/qiskit.transpiler.Target) berisi semua batasan yang relevan dari sebuah perangkat, seperti basis Gate native-nya, konektivitas Qubit, dan informasi pulse atau timing.
- [`Backend`](../api/qiskit/qiskit.providers.BackendV2) memiliki `Target` secara default, berisi informasi tambahan -- seperti [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap), dan menyediakan antarmuka untuk mengirimkan job Circuit kuantum.

Kamu juga bisa secara eksplisit memberikan informasi untuk digunakan transpiler, misalnya jika kamu punya kasus penggunaan tertentu, atau jika kamu yakin informasi ini akan membantu transpiler menghasilkan Circuit yang lebih optimal.

Ketepatan transpiler dalam menghasilkan Circuit yang paling sesuai untuk hardware tertentu bergantung pada seberapa banyak informasi yang dimiliki `Target` atau `Backend` tentang batasannya.

> **Note:** Karena banyak algoritma transpilasi yang bersifat stokastik, tidak ada jaminan bahwa Circuit yang lebih baik akan ditemukan.

Halaman ini menunjukkan beberapa contoh pengiriman informasi QPU ke transpiler. Contoh-contoh ini menggunakan target dari backend mock [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

<span id="default-config"></span>
## Konfigurasi default
Penggunaan transpiler yang paling sederhana adalah memberikan semua informasi QPU dengan menyediakan `Backend` atau `Target`. Untuk lebih memahami cara kerja transpiler, buat sebuah Circuit dan transpilasikan dengan informasi yang berbeda, seperti berikut.

Impor pustaka yang diperlukan dan buat instansi QPU:
Untuk mengonversi Circuit abstrak menjadi ISA circuit yang bisa dijalankan di prosesor tertentu, transpiler membutuhkan informasi tertentu tentang prosesor tersebut. Biasanya, informasi ini disimpan di [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) atau [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) yang diberikan ke transpiler, dan tidak diperlukan informasi lebih lanjut. Namun, kamu juga bisa secara eksplisit memberikan informasi untuk digunakan transpiler, misalnya jika kamu punya kasus penggunaan tertentu, atau jika kamu yakin informasi ini akan membantu transpiler menghasilkan Circuit yang lebih optimal.

Topik ini menunjukkan beberapa contoh pengiriman informasi ke transpiler. Contoh-contoh ini menggunakan target dari backend mock [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

Circuit contoh menggunakan instansi dari [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) dari pustaka Circuit Qiskit.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

Contoh ini menggunakan pengaturan default untuk mentranspilasi ke `target` dari `backend`, yang menyediakan semua informasi yang diperlukan untuk mengonversi Circuit menjadi Circuit yang bisa dijalankan di backend.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

Contoh ini digunakan di bagian-bagian selanjutnya dari topik ini untuk mengilustrasikan bahwa coupling map dan basis Gate adalah informasi esensial yang perlu diberikan ke transpiler untuk konstruksi Circuit yang optimal. QPU biasanya bisa memilih pengaturan default untuk informasi lain yang tidak diberikan, seperti timing dan scheduling.

## Coupling map

Coupling map adalah graf yang menunjukkan Qubit mana yang terhubung dan karenanya memiliki Gate dua Qubit di antara mereka. Terkadang graf ini bersifat terarah, artinya Gate dua Qubit hanya bisa berjalan dalam satu arah. Namun, transpiler selalu bisa membalik arah Gate dengan menambahkan Gate satu Qubit tambahan. Circuit kuantum abstrak selalu bisa direpresentasikan pada graf ini, meskipun konektivitasnya terbatas, dengan memperkenalkan Gate SWAP untuk memindahkan informasi kuantum.

Qubit dari Circuit abstrak kita disebut _Qubit virtual_ dan yang ada di coupling map disebut _Qubit fisik_. Transpiler menyediakan pemetaan antara Qubit virtual dan fisik. Salah satu langkah pertama dalam transpilasi, tahap _layout_, melakukan pemetaan ini.

> **Note:** Meskipun tahap routing saling terkait dengan tahap _layout_ — yang memilih Qubit yang sebenarnya — secara default, topik ini memperlakukannya sebagai tahap terpisah untuk kesederhanaan. Kombinasi routing dan layout disebut _pemetaan Qubit_. Pelajari lebih lanjut tentang tahap-tahap ini di topik [Tahap Transpiler](transpiler-stages).

Berikan argumen kata kunci `coupling_map` untuk melihat efeknya pada transpiler:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

Seperti yang ditunjukkan di atas, beberapa Gate SWAP dimasukkan (masing-masing terdiri dari tiga Gate CX), yang akan menyebabkan banyak error di perangkat saat ini. Untuk melihat Qubit mana yang dipilih pada topologi Qubit yang sebenarnya, gunakan `plot_circuit_layout` dari Visualisasi Qiskit:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

Ini menunjukkan bahwa Qubit virtual 0-11 kita dipetakan secara trivial ke baris Qubit fisik 0-11. Mari kita kembali ke default (`optimization_level=1`), yang menggunakan `VF2Layout` jika diperlukan routing.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

Sekarang tidak ada Gate SWAP yang dimasukkan dan Qubit fisik yang dipilih sama seperti saat menggunakan kelas `target`.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

Sekarang layoutnya berbentuk cincin. Karena layout ini menghormati konektivitas Circuit, tidak ada Gate SWAP, menghasilkan Circuit yang jauh lebih baik untuk eksekusi.

## Basis Gate

Setiap komputer kuantum mendukung set instruksi terbatas, yang disebut _basis Gate_-nya. Setiap Gate dalam Circuit harus diterjemahkan ke elemen-elemen set ini. Set ini harus terdiri dari Gate satu dan dua Qubit yang menyediakan set Gate universal, artinya operasi kuantum apa pun bisa didekomposisi menjadi Gate-Gate tersebut. Ini dilakukan oleh [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator), dan basis Gate bisa ditentukan sebagai argumen kata kunci ke transpiler untuk memberikan informasi ini.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

Gate satu Qubit default di _ibm_sherbrooke_ adalah `rz`, `x`, dan `sx`, dan Gate dua Qubit default adalah `ecr` (echoed cross-resonance). Gate CX dibangun dari Gate `ecr`, jadi di beberapa QPU `ecr` ditentukan sebagai basis Gate dua Qubit, sementara di QPU lain `cx` adalah defaultnya. Gate `ecr` adalah bagian _entangling_ dari Gate `cx`. Selain Gate kontrol, ada juga instruksi `delay` dan `measurement`.

> **Note:** QPU memiliki basis Gate default, tapi kamu bisa memilih Gate apa pun yang kamu mau, selama kamu menyediakan instruksi atau menambahkan pulse Gate (lihat [Membuat transpiler pass.](custom-transpiler-pass)) Basis Gate default adalah yang kalibrasinya telah dilakukan di QPU, sehingga tidak perlu memberikan instruksi/pulse Gate lebih lanjut. Misalnya, di beberapa QPU `cx` adalah Gate dua Qubit default dan `ecr` di QPU lainnya. Lihat daftar [Gate dan operasi native yang mungkin](/guides/qpu-information#native-gates) untuk detail lebih lanjut.

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

Perhatikan bahwa objek `CXGate` telah didekomposisi menjadi Gate `ecr` dan basis Gate satu Qubit.
## Tingkat error perangkat
Kelas `Target` bisa berisi informasi tentang tingkat error untuk operasi di perangkat.
Misalnya, kode berikut mengambil properti untuk Gate echoed cross-resonance (ECR) antara Qubit 1 dan 0 (perhatikan bahwa Gate ECR bersifat terarah):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

Output menampilkan durasi Gate (dalam detik) dan tingkat error-nya. Untuk mengungkapkan informasi error ke transpiler, buat model target dengan `basis_gates` dan `coupling_map` dari atas dan isi dengan nilai error dari backend `FakeSherbrooke`.